# Init - Import Libraries

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import StringType, DateType

# Reading from bronze

In [0]:
df = spark.table('workspace.bronze.crm_prd_info')

In [0]:
df.limit(10).display()

In [0]:
df.printSchema()

# Product Key Parsing - padronização do product key

In [0]:
# df2 = df
df = df.withColumn('cat_id', F.regexp_replace(F.substring(F.col('prd_key'), 1, 5), '-', '_'))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))


# Renaming columns - Renomeando colunas

In [0]:
RENAME_MAP = {
    "prd_id" : "product_id"
    , 'cat_id': 'category_id'
    , "prd_key": "product_key"
    , "prd_nm": "product_name"
    , "prd_cost": "product_cost"
    , "prd_line": "product_line"	
    , "prd_start_dt": "product_start_date"
    , "prd_end_dt": "product_end_date"
}

In [0]:
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Trimming values

In [0]:
for i in df.schema.fields:
    # print(i.name, i.dataType)
    if isinstance(i.dataType, StringType):
        df = df.withColumn(i.name, trim(col(i.name)))

# Cost cleanup - limpeza do valor do produto

In [0]:
df = df.withColumn("product_cost", F.coalesce(col("product_cost"), F.lit(0)))

# Product Line Normalization - Normalização da coluna Product Line

In [0]:

df = (
    df
    # Normalize product line
    .withColumn(
        "product_line",
        F.when(F.upper(col("product_line")) == "M", "Mountain")
         .when(F.upper(col("product_line")) == "R", "Road")
         .when(F.upper(col("product_line")) == "S", "Other Sales")
         .when(F.upper(col("product_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)

# Date Casting

In [0]:
df = df.withColumn("product_start_date", col("product_start_date").cast(DateType()))

# Writing silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")